# Magic Card To Text Dataset Generator  
__Objective:__ The aim of this notebook is to harness the IBM Granite Docling model to convert a large batch of Magic the Gathering card images into a structured text dataset. That dataset can then be used to fine tune a transformer model on multi-label classification of scryfall tags.

In [ ]:
# config
MODEL_NAME = "ibm-granite/granite-docling-258M"
MAX_SEQUENCE_LENGTH = 256
BATCH_SIZE = 1 # even two may spike RAM

# CARD_EXTRACTION_PROMPT = 'Read every piece of visible text from this card.'
# CARD_EXTRACTION_PROMPT = 'Transcribe this Magic card exactly.'
CARD_EXTRACTION_PROMPT = '''
Transcribe this Magic card as plain text.
Do not include bounding boxes, coordinates, locations, XML tags, or markup.
'''

## Packages and Data

In [2]:
# packages

## link directory
from pathlib import Path
import sys

workspace_root = Path.cwd()
if workspace_root.name == 'notebooks':
    workspace_root = workspace_root.parent
if str(workspace_root) not in sys.path:
    sys.path.append(str(workspace_root))

## custom packages
from src.card_ocr.dataset import load_manifest_records, summarize_manifest

In [3]:
# retrieve core data
manifest_path = workspace_root / "data" / "card_image_text_manifest.jsonl"
records = load_manifest_records(manifest_path, skip_missing_images=True)
summary = summarize_manifest(records)
summary


{'num_records': 8443, 'split_counts': {'train': 6713, 'val': 1680, 'test': 50}}

In [4]:
records[0]

{'id': 8079,
 'oracle_id': '4317f4f1-d339-4940-b46e-659641035595',
 'card_name': 'Ascended Lawmage',
 'split': 'train',
 'image_path': '/Users/nickcruickshank/Projects/scryfall-llm-sandbox/data/card_images/4317f4f1-d339-4940-b46e-659641035595.png',
 'image_type': 'png',
 'image_exists': True,
 'target_text': "Ascended Lawmage\n        Mana Cost = {2}{W}{U}\nMana Value = 4.0\n\n        Type Line = Creature — Vedalken Wizard\n\n        Rules Text = Flying\nHexproof (This creature can't be the target of spells or abilities your opponents control.)\n\n        Power = 3\nToughness = 2\n\n\n        Color Identity = ['U', 'W']\n\n        Rarity = uncommon",
 'tags': ['evasion', 'french vanilla']}

## Prepare Image Dataset

In [5]:
# import torch
from torch.utils.data import Dataset
from PIL import Image

class GraniteCardOCRDataset(Dataset):
    """
    Description
    ----------
    This class contains the dataset for the Granite OCR model which we will use 
    to extract the text from Magic the Gathering cards.

    Inputs
    ----------
    records = A list of dicts containing our dataset elements
    resize_height = The height to resize the images to. Defaultt to 384 pixels.
        Can likely acchieve higher quality resolution with up to 512 pixels,
        but would cost more compute.
    """
    def __init__(self, records, resize_height:int = 384):
        self.records = records
        self.resize_height = resize_height

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        # get the record 
        record = self.records[index]

        # retrieve the image and preprocess it
        with Image.open(record['image_path']) as image:
            # quick process the image
            image = image.convert('RGB')
            image = image.resize(
                (
                    int(image.width * self.resize_height / image.height),
                    self.resize_height
                )
            )

            # create output
            out = {
                'id': record['id'],
                'oracle_id': record['oracle_id'],
                'card_name': record['card_name'],
                'image': image,
                'split': record['split']
            }

            return out

In [6]:
class GraniteCardOCRCollator:
    """ 
    Description
    ----------
    This class is used to collate the dataset for the Granite OCR model which we will use
    downstream to translate generated text into scryfall tags.

    Inputs
    ----------
    processor = The Granite OCR processor which will be used to collate the dataset.
    """
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):
        # extract the batch images
        images = [x['image'] for x in batch]

        # send to processor
        inputs = self.processor(
            images = images,
            return_tensors = 'pt'
        )

        inputs['ids'] = [x['id'] for x in batch]
        inputs['oracle_ids'] = [x['oracle_id'] for x in batch]
        inputs['card_names'] = [x['card_name'] for x in batch]
        inputs['splits'] = [x['split'] for x in batch]

        return inputs

In [7]:
train_records = [record for record in records if record['split'] == 'train']
val_records = [record for record in records if record['split'] == 'val']
test_records = [record for record in records if record['split'] == 'test']
print(f'Train: {len(train_records)} | Test: {len(test_records)} | Val: {len(val_records)}')

Train: 6713 | Test: 50 | Val: 1680


In [8]:
# prepare datasets
from torch.utils.data import DataLoader
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(MODEL_NAME)
img_dataset = GraniteCardOCRDataset(test_records)
img_dataloader = DataLoader(
    dataset = img_dataset,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = 0,    # recommended for macoS
    pin_memory = False, # MPS doesn't benefit from pin_memory
    collate_fn = GraniteCardOCRCollator(processor)
)
dict(next(iter(img_dataloader))).keys()

dict_keys(['pixel_values', 'pixel_attention_mask', 'rows', 'cols', 'ids', 'oracle_ids', 'card_names', 'splits'])

## Get Model

In [9]:
import torch

def get_device_and_dtype():
    if torch.backends.mps.is_available():
        return torch.device("mps"), torch.float16
    if torch.cuda.is_available():
        return torch.device("cuda"), torch.float16
    return torch.device("cpu"), torch.float32

baseline_device, baseline_dtype = get_device_and_dtype()
baseline_device, baseline_dtype

(device(type='mps'), torch.float16)

In [10]:

from transformers import AutoModelForVision2Seq
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=baseline_dtype,
)
model.to(baseline_device)
model.eval()

/Users/nickcruickshank/Projects/scryfall-llm-sandbox/.venv/lib/python3.14/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Idefics3ForConditionalGeneration(
  (model): Idefics3Model(
    (vision_model): Idefics3VisionTransformer(
      (embeddings): Idefics3VisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
        (position_embedding): Embedding(1024, 768)
      )
      (encoder): Idefics3Encoder(
        (layers): ModuleList(
          (0-11): 12 x Idefics3EncoderLayer(
            (self_attn): Idefics3VisionAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
            (mlp): Idefics3VisionMLP(
              (activation_fn): GELUTanh()
              (fc1): Linear(in_featur

## Process Images Into Text

In [ ]:
import torch
from tqdm.auto import tqdm

results = []

with torch.inference_mode():

    for batch in tqdm(img_dataloader, desc="Processing Batches"):
        # get batch info
        ids = batch.pop('ids')
        oracle_ids = batch.pop('oracle_ids')
        card_names = batch.pop('card_names')
        splits = batch.pop('splits')

        # move tenosrs to model device
        allowed_keys = {"pixel_values", "pixel_attention_mask"}
        batch = {
            k: v.to(model.device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
            if k in allowed_keys
        }

        # create outputs (raw)
        output_ids = model.generate(
            **batch,
            max_new_tokens = MAX_SEQUENCE_LENGTH,
            do_sample = False
        )

        # reshape raw outputs to text
        generated_text = processor.batch_decode(
            output_ids,
            skip_special_tokens = True
        )

        # store results
        for card_id, oracle_id, card_name, text, split in zip(
            ids, oracle_ids, card_names, generated_text, splits
        ):
            results.append(
                {
                    'id': card_id,
                    'oracle_id': oracle_id,
                    'card_name': card_name,
                    'text': generated_text.strip(),
                    'split': split
                }
            )

results

Processing Batches:   0%|          | 0/25 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


## Graveyard

In [ ]:
# def get_tokenizer(processor):
#     if hasattr(processor, 'tokenizer'):
#         return processor.tokenizer
#     raise AttributeError('Expected the Granite processor to expose a tokenizer')

In [ ]:
# def apply_chat_template(
#     processor,
#     messages, 
#     add_generation_prompt:bool = False 
# ):    
#     return processor.apply_chat_template(
#         messages,
#         add_generation_prompt = add_generation_prompt,
#         tokenize = False
#     )

# def build_messages(
#     prompt:str,
#     target_text = None
# ):
#     messages = [
#         {
#             'role': 'user',
#             'content': [
#                 {'type': 'image'},
#                 {'type': 'text', 'text': prompt}
#             ]
#         }
#     ]

#     if target_text is not None:
#         messages.append(
#             {
#                 'role': 'assistant',
#                 'content': [{'type': 'text', 'text': target_text}]
#             }
#         )

#     return messages

In [ ]:
# class GraniteCardOCRCollator:
#     """
#     Description
#     ----------
#     This class is used to collate the dataset elements into batches for training.

#     Inputs
#     ----------

#     """
#     def __init__(
#         self, 
#         processor, 
#         tokenizer,
#         prompt, 
#         max_sequence_length:int = 256
#     ):
#         self.processor = processor
#         self.tokenizer = tokenizer
#         self.prompt = prompt
#         self.max_sequence_length = max_sequence_length
#         self.tokenizer = get_tokenizer(processor)

#     def __call__(self, features):
#         images = [feature['image'] for feature in features]

#         prompt_texts = [
#             apply_chat_template(
#                 self.processor,
#                 build_messages(self.prompt),
#                 add_generation_prompt = True
#             )
#             for _ in features
#         ]

#         # deliberately skipping the full_texts part, as we want to make text from images

#         batch = self.processor(
#             images = images,
#             padding = True,
#             truncation = False,
#             max_length = self.max_sequence_length,
#             return_tensors = 'pt'
#         )
#         labels = batch['input_ids'].clone()

#         for row_idx, prompt_text in enumerate(prompt_texts):
#             prompt_ids = self.tokenizer(
#                 prompt_text,
#                 add_special_tokens = False,
#                 truncation = False,
#                 max_length = self.max_sequence_length
#             ).input_ids 

#             prompt_len = min(len(prompt_ids), labels.shape[1])
#             labels[row_idx, :prompt_len] = -100 

#         if self.tokenizer.pad_token_id is not None:
#             labels[labels == self.tokenizer.pad_token_id] = -100 

#         batch['labels'] = labels
#         return batch



In [ ]:
# from transformers import AutoProcessor

# processor = AutoProcessor.from_pretrained(MODEL_NAME)
# tokenizer = get_tokenizer(processor)

# collator = GraniteCardOCRCollator(
#     processor = processor,
#     tokenizer = tokenizer,
#     max_sequence_length = MAX_SEQUENCE_LENGTH,
#     prompt = CARD_EXTRACTION_PROMPT
# )